# Download Models

In [ ]:
!apt-get update && apt-get -y install -qq aria2

import os
import urllib
from pathlib import Path

class Aria2Builder:
    class UrlEntry:
        def __init__(self, url: str, subdir: str, out: str, headers: dict[str, str]):
            self.url = url
            self.subdir = subdir
            self.out: str | None = out
            self.headers = headers

    def reset(self) -> None:
        self._comfyui_models_dir: Path = Path("ComfyUI") / "models"
        self._huggingface_token = os.environ.get("HUGGINGFACE_TOKEN")
        self._civitai_token = os.environ.get("CIVITAI_TOKEN")
        self._file_path: Path = Path("aria2_input_file.txt")
        self._url_entries: list[Aria2Builder.UrlEntry] = []

    def __init__(
        self,
    ):
        self.reset()

    def comfyui_models_dir(self, comfyui_models_dir: Path) -> None:
        self._comfyui_models_dir = comfyui_models_dir

    def huggingface_token(self, huggingface_token: str) -> None:
        self._huggingface_token = huggingface_token

    def civitai_token(self, civitai_token: str) -> None:
        self._civitai_token = civitai_token

    def file_path(self, file_path: Path) -> None:
        self._file_path = file_path

    def add_url(self, url: str, subdir: str) -> None:
        parsed_url: urllib.parse.ParseResult = urllib.parse.urlparse(url)

        # Authentication
        headers = {}
        if "huggingface.co" in parsed_url.netloc:
            if not self._huggingface_token:
                raise ValueError(
                    "huggingface_token is not set and not specified in HUGGINGFACE_TOKEN environment variable."
                )
            headers["Authorization"] = f"Bearer {self._huggingface_token}"
            # Huggingface has the filename in the url. The files will break if we don't explicitly set the output filename

        elif "civitai.com" in parsed_url.netloc or "civitai.red" in parsed_url.netloc:
            if not self._civitai_token:
                raise ValueError(
                    "civitai_token is not set and not specified in CIVITAI_TOKEN environment variable."
                )
            url = f"{url}&token={self._civitai_token}"

        
        # Fix filenames for repositories that screw up the filename if not set explicitly
        out = None
        if "huggingface.co" in parsed_url.netloc or "github.com" in parsed_url.netloc:
            out = os.path.basename(parsed_url.path)
            if not out:
                raise ValueError("Could not determine filename from URL for repository that requires it to be set explicitly.")

        url_entry = Aria2Builder.UrlEntry(url, subdir, out, headers)
        self._url_entries.append(url_entry)

    def build(self) -> None:
        boilerplate_options = {
            "split": "16",
            "max-connection-per-server": "16",
            "min-split-size": "1M",
            "allow-overwrite": "true",
            "continue": "true",
            "auto-file-renaming": "false",
        }
        with open(self._file_path, "w") as file:
            for url_entry in self._url_entries:
                options = {
                    "dir": str(self._comfyui_models_dir / url_entry.subdir),
                }
                if url_entry.out:
                    options["out"] = url_entry.out
                if url_entry.headers:
                    options["header"] = "\n".join([
                        f"{k}: {v}" for k, v in url_entry.headers.items()
                    ])

                file.write(f"{url_entry.url}\n")
                for option, value in boilerplate_options.items():
                    file.write(f"\t{option}={value}\n")
                for option, value in options.items():
                    file.write(f"\t{option}={value}\n")
        self.reset()


## Upscalers

In [ ]:
aria2_builder = Aria2Builder()
aria2_builder.file_path(Path("upscalers_aria2.txt"))

# --- ADD MODELS TO DOWNLOAD HERE ---
aria2_builder.add_url(
    "https://civitai.com/api/download/models/125843?type=Model&format=PickleTensor", 
    "upscale_models",
)

aria2_builder.add_url(
    "https://github.com/Phhofm/models/releases/download/2xNomosUni_span_multijpg_ldl/2xNomosUni_span_multijpg_ldl.safetensors",
    "upscale_models",
)

aria2_builder.add_url(
    "https://github.com/Phhofm/models/releases/download/4xNomos8k_atd_jpg/4xNomos8k_atd_jpg.safetensors",
    "upscale_models",
)
# -----------------------------------

# Start download
aria2_builder.build()
!aria2c  --console-log-level=error -i upscalers_aria2.txt

## Hunyuan Models

In [ ]:
aria2_builder = Aria2Builder()
aria2_builder.file_path(Path("hunyuan_aria2.txt"))

# --- ADD MODELS TO DOWNLOAD HERE ---
# HunyuanVideo models
aria2_builder.add_url(
    "https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/resolve/main/split_files/vae/hunyuan_video_vae_bf16.safetensors",
    "vae",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1356617?type=Model&format=SafeTensor&size=pruned&fp=fp8",
    "diffusion_models",
)
aria2_builder.add_url(
    "https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/resolve/main/split_files/text_encoders/llava_llama3_fp8_scaled.safetensors",
    "text_encoders",
)
# Additional text encoder
aria2_builder.add_url(
    "https://huggingface.co/zer0int/LongCLIP-SAE-ViT-L-14/resolve/main/Long-ViT-L-14-GmP-SAE-TE-only.safetensors",
    "text_encoders",
)
# HunyuanVideo Fast Model
aria2_builder.add_url(
    "https://huggingface.co/Kijai/HunyuanVideo_comfy/resolve/main/hunyuan_video_FastVideo_720_fp8_e4m3fn.safetensors",
    "diffusion_models",
)


# HunyuanVideo LoRAs
# Hunyuan FastVideo LoRA  by Kijai
aria2_builder.add_url(
    "https://huggingface.co/Kijai/HunyuanVideo_comfy/resolve/main/hyvideo_FastVideo_LoRA-fp8.safetensors",
    "loras",
)
# Hunyuan I2V
aria2_builder.add_url(
    "https://huggingface.co/leapfusion-image2vid-test/image2vid-960x544/resolve/main/img2vid544p.safetensors",
    "loras",
)

# Female Masturbation
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1259737?type=Model&format=SafeTensor",
    "loras",
)
# Riding Dildo
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1261435?type=Model&format=SafeTensor",
    "loras",
)
# Edge of Reality
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1270232?type=Model&format=SafeTensor",
    "loras",
)
# Cum on Face
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1231959?type=Model&format=SafeTensor",
    "loras",
)
# Undressing
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1299285?type=Model&format=SafeTensor",
    "loras",
)
# Missionary
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1385168?type=Model&format=SafeTensor",
    "loras",
)
# Cumshot
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1289279?type=Model&format=SafeTensor",
    "loras",
)
# Doggystyle from behind
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1239432?type=Model&format=SafeTensor",
    "loras",
)
# Cowgirl POV
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1187802?type=Model&format=SafeTensor",
    "loras",
)
# PiV motion helper
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1419218?type=Model&format=SafeTensor",
    "loras",
)
# PiV motion helper PoV v0.4
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1435515?type=Model&format=SafeTensor",
    "loras",
)
# Nipple play
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1291865?type=Model&format=SafeTensor",
    "loras",
)
# Breast massage
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1410507?type=Model&format=SafeTensor",
    "loras",
)
# Oral sex
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1188578?type=Model&format=SafeTensor",
    "loras",
)
# Dance
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1367561?type=Model&format=SafeTensor",
    "loras"
)
# Disney
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1389959?type=Model&format=SafeTensor",
    "loras"
)

# Porn salt
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1501799?type=Model&format=SafeTensor",
    "loras"
)

# O face
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1497241?type=Model&format=SafeTensor",
    "loras"
)





# -----------------------------------

# Start download
aria2_builder.build()
!aria2c  --console-log-level=error -i hunyuan_aria2.txt

## Flux

In [ ]:
aria2_builder = Aria2Builder()
aria2_builder.file_path(Path("flux_aria2.txt"))

# --- ADD MODELS TO DOWNLOAD HERE ---
# flux-dev GGUF Q8_0
aria2_builder.add_url(
    "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/flux1-dev-Q8_0.gguf",
    "unet",
)

# # flux-dev fp8
# aria2_builder.add_url(
#     "https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-dev-fp8.safetensors",
#     "unet",
# )

# # flux-dev fp16
# aria2_builder.add_url(
#     "https://civitai.com/api/download/models/691639?type=Model&format=SafeTensor&size=full&fp=fp32",
#     "unet",
# )

# # flux-krea-dev fp8
# aria2_builder.add_url(
#     "https://civitai.com/api/download/models/2068069?type=Model&format=SafeTensor&size=pruned&fp=fp8",
#     "unet",
# )

# unStable Evolution Krea GGUF Q8_0 https://civitai.com/models/1931032?modelVersionId=2274229
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2274229?type=Model&format=GGUF&size=pruned&fp=fp16",
    "unet",
)

# # FASCIUM.KREA.FLUX.NSFW v8 (fp8)
# aria2_builder.add_url(
#     "https://civitai.com/api/download/models/2293408?type=Model&format=SafeTensor&size=pruned&fp=fp8",
#     "unet",
# )
# # FASCIUM.KREA.FLUX.NSFW v7 (fp8)
# aria2_builder.add_url(
#     "https://civitai.com/api/download/models/2217591?type=Model&format=SafeTensor&size=pruned&fp=fp8",
#     "unet",
# )




# vae
aria2_builder.add_url(
    "https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/ae.safetensors",
    "vae",
)

# clip 
aria2_builder.add_url(
    "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors",
    "clip",
)
aria2_builder.add_url(
    "https://huggingface.co/zer0int/CLIP-GmP-ViT-L-14/resolve/main/ViT-L-14-TEXT-detail-improved-hiT-GmP-TE-only-HF.safetensors",
    "clip",
)
aria2_builder.add_url(
    "https://huggingface.co/zer0int/CLIP-GmP-ViT-L-14/resolve/main/ViT-L-14-BEST-smooth-GmP-TE-only-HF-format.safetensors",
    "clip",
)
aria2_builder.add_url(
    "https://huggingface.co/zer0int/CLIP-Registers-Gated_MLP-ViT-L-14/resolve/main/ViT-L-14-REG-TE-only-balanced-HF-format-ckpt12.safetensors",
    "clip",
)




# # T5 model fp8
# aria2_builder.add_url(
#     "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors",
#     "clip",
# )

# # T5 model Q8_0 bit
# aria2_builder.add_url(
#     "https://huggingface.co/city96/t5-v1_1-xxl-encoder-gguf/resolve/main/t5-v1_1-xxl-encoder-Q8_0.gguf",
#     "clip",
# )

# T5 model fp16
aria2_builder.add_url(
    "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors",
    "clip",
)

# LoRAs
# Speed LoRA
aria2_builder.add_url(
    "https://huggingface.co/ByteDance/Hyper-SD/resolve/main/Hyper-FLUX.1-dev-8steps-lora.safetensors",
    "loras",
)


# Style LORAS
aria2_builder.add_url(
    "https://civitai.com/api/download/models/736227?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1047380?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1500495?type=Model&format=SafeTensor",
    "loras",
)
# NSFW master strength 0.5 - 0.8
aria2_builder.add_url(
    "https://civitai.com/api/download/models/746602?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/931225?type=Model&format=SafeTensor",
    "loras",
)
# aidmarealisticskin strengh 0.6
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1301668?type=Model&format=SafeTensor",
    "loras",
)
# Undressing
aria2_builder.add_url(
    "https://civitai.com/api/download/models/917520?type=Model&format=SafeTensor",
    "loras",
)

# JCTits strength 0.5 (?) with NSWFmaster 0.5
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1321842?type=Model&format=SafeTensor",
    "loras",
)
# EGTits strength 0.5  with NSWFmaster 0.5
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1169319?type=Model&format=SafeTensor",
    "loras",
)
# KUTits strength 0.75  with NSWFmaster 0.5
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1278213?type=Model&format=SafeTensor",
    "loras",
)
# AdATitties strength 1.0 (?) with NSWFmaster 0.5
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1093128?type=Model&format=SafeTensor",
    "loras",
)

# Nipple diffusion General strenth 1.0
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1047380?type=Model&format=SafeTensor",
    "loras",
)
# Nipple diffusion Bumpy strenth 1.0
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1867123?type=Model&format=SafeTensor",
    "loras",
)
# Nipple diffusion Wrinkled strenth 1.0
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1867163?type=Model&format=SafeTensor",
    "loras",
)
# Nipple diffusion Oval strenth 1.0
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1865751?type=Model&format=SafeTensor",
    "loras",
)
# Nipple diffusion Flat strenth 1.0
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1871038?type=Model&format=SafeTensor",
    "loras",
)
# Nipple diffusion Long strenth 1.0 
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1064546?type=Model&format=SafeTensor",
    "loras",
)
# Nipple diffusion Puffy strenth 1.0
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1069819?type=Model&format=SafeTensor",
    "loras",
)
# Nipple diffusion Ghost strenth 1.0
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1062916?type=Model&format=SafeTensor",
    "loras",
)
# Nipple diffusion Small strenth 1.0
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1068253?type=Model&format=SafeTensor",
    "loras",
)
# Nipple Diffusion Big strengrth 1.0
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1066495?type=Model&format=SafeTensor",
    "loras",
)

# Ink style zydInk strength 0.8-1.2
aria2_builder.add_url(
    "https://civitai.com/api/download/models/890482?type=Model&format=SafeTensor",
    "loras",
)

# Ink wash fusion "A dinkfsn style ink wash painting" strength 0.8-1.2
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1524366?type=Model&format=SafeTensor",
    "loras",
)

# Female masturbation
aria2_builder.add_url(
    "https://civitai.com/api/download/models/928767?type=Model&format=SafeTensor",
    "loras",
)

# Mystic-XXX-v7 NSFW
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2009929?type=Model&format=SafeTensor",
    "loras",
)

# Mystic Hentai
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1892397?type=Model&format=SafeTensor",
    "loras",
)

# RiMix Realistic illustration style
aria2_builder.add_url(
    "https://civitai.com/api/download/models/1918677?type=Model&format=SafeTensor",
    "loras",
)


# Ghibli style ghiblistyle
aria2_builder.add_url(
    "https://civitai.com/api/download/models/755852?type=Model&format=SafeTensor",
    "loras",
)



# -----------------------------------

# Start download
aria2_builder.build()
!aria2c  --console-log-level=error -i flux_aria2.txt

## Pony

In [ ]:
aria2_builder = Aria2Builder()
aria2_builder.file_path(Path("pony_aria2.txt"))

# --- ADD MODELS TO DOWNLOAD HERE ---

# # Pony Diffusion V6 XL
# aria2_builder.add_url(
#     "https://civitai.com/api/download/models/290640?type=Model&format=SafeTensor&size=pruned&fp=fp16",
#     "checkpoints",
# )
# vae
# aria2_builder.add_url(
#     "https://civitai.com/api/download/models/290640?type=VAE&format=SafeTensor",
#     "vae",
# )
# # Pony Realism
# aria2_builder.add_url(
#     "https://civitai.com/api/download/models/914390?type=Model&format=SafeTensor&size=full&fp=fp16",
#     "checkpoints",
# )
# Autism mix confetti
aria2_builder.add_url(
    "https://civitai.com/api/download/models/324524?type=Model&format=SafeTensor&size=pruned&fp=fp16",
    "checkpoints",
)

# LORAS
# Speed LORAS
aria2_builder.add_url(
    "https://huggingface.co/wangfuyun/PCM_Weights/resolve/main/sdxl/pcm_sdxl_normalcfg_8step_converted.safetensors",
    "loras",
)
aria2_builder.add_url(
    "https://huggingface.co/wangfuyun/PCM_Weights/resolve/main/sdxl/pcm_sdxl_normalcfg_16step_converted.safetensors",
    "loras",
)

# Style LORAS
aria2_builder.add_url(
    "https://civitai.com/api/download/models/418782?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/450029?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/418769?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/398292?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/372898?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/363388?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/341131?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/333607?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/333590?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/333587?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/329446?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/323081?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/302106?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/300686?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/298238?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/298005?type=Model&format=SafeTensor",
    "loras",
)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/297988?type=Model&format=SafeTensor",
    "loras",
)
# 50 artist styles lora
aria2_builder.add_url(
    "https://civitai.com/api/download/models/369272?type=Model&format=SafeTensor",
    "loras",
)

# Embeddings
# zPDXLxxx
aria2_builder.add_url(
    "https://civitai.com/api/download/models/380277?type=Model&format=PickleTensor",
    "embeddings",
)
# zPDXLxxx-neg
aria2_builder.add_url(
    "https://civitai.com/api/download/models/380277?type=Negative&format=Other",
    "embeddings",
)
# zPDXLrl
aria2_builder.add_url(
    "https://civitai.com/api/download/models/482268?type=Model&format=PickleTensor",
    "embeddings",
) 
# zPDXLrl-neg
aria2_builder.add_url(
    "https://civitai.com/api/download/models/482268?type=Negative&format=Other",
    "embeddings",
)
# zPDXL3 (High quality)
aria2_builder.add_url(
    "https://civitai.com/api/download/models/720175?type=Model&format=SafeTensor",
    "embeddings",
)


# -----------------------------------

# Start download
aria2_builder.build()
!aria2c  --console-log-level=error -i pony_aria2.txt

# Z image

In [ ]:
aria2_builder = Aria2Builder()
aria2_builder.file_path(Path("zimage_aria2.txt"))

# --- ADD MODELS TO DOWNLOAD HERE ---

# Diffusion model
aria2_builder.add_url(
    "https://huggingface.co/Comfy-Org/z_image_turbo/blob/main/split_files/diffusion_models/z_image_turbo_bf16.safetensors",
    "diffusion_models",
)

# VAE
aria2_builder.add_url(
    "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors",
    "vae",
)

# Text encoder
aria2_builder.add_url(
    "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/text_encoder.safetensors",
    "text_encoders",
)

# LORAS
# Speed LORAS
aria2_builder.add_url(
    "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/loras/z_image_turbo_distill_patch_lora_bf16.safetensors",
    "loras",
)


# Style LORAS
# Photo real better Nudes https://civitai.com/models/2174081/photoreal-betternudes-nsfw?modelVersionId=2474435
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2474435?type=Model&format=SafeTensor",
    "loras",
)
# Mystic XXX https://civitai.com/models/2206377/zit-mystic-xxx?modelVersionId=2581135
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2581135?type=Model&format=SafeTensor",
    "loras",
)
# Jib Mix https://civitai.com/models/2194714?modelVersionId=2471161
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2471161?type=Model&format=SafeTensor",
    "loras",
)
# -----------------------------------

# Start download
aria2_builder.build()
!aria2c  --console-log-level=error -i zimage_aria2.txt

# Chroma

In [ ]:
aria2_builder = Aria2Builder()
aria2_builder.file_path(Path("chroma_aria2.txt"))

# --- ADD MODELS TO DOWNLOAD HERE ---
# flux-dev GGUF Q8_0

# Chroma v5.0
aria2_builder.add_url(
    "https://huggingface.co/lodestones/Chroma/resolve/main/chroma-unlocked-v50.safetensors",
    "unet",
)

# vae
aria2_builder.add_url(
    "https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/ae.safetensors",
    "vae",
)

# T5 model fp16
aria2_builder.add_url(
    "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors",
    "clip",
)

# Chroma LoRA
aria2_builder.add_url(
    "https://huggingface.co/silveroxides/Chroma-LoRA-Experiments/resolve/main/chroma-unlocked-rescaled_cfg_LoRA-rank_16-fp32.safetensors",
    "loras",
)


# -----------------------------------

# Start download
aria2_builder.build()
!aria2c  --console-log-level=error -i chroma_aria2.txt

# QWEN image

In [ ]:
aria2_builder = Aria2Builder()
aria2_builder.file_path(Path("qwen_image_aria2.txt"))

# --- ADD MODELS TO DOWNLOAD HERE ---

#  QWEN image diffusionmodel
aria2_builder.add_url(
    "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/diffusion_models/qwen_image_fp8_e4m3fn.safetensors",
    "diffusion_models",
)

# vae
aria2_builder.add_url(
    "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors",
    "vae",
)

# Text encoder
aria2_builder.add_url(
    "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors",
    "text_encoders",
)

# Loras
# Mystic XXX NSFW
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2195978?type=Model&format=SafeTensor",
    "loras",
)

# -----------------------------------

# Start download
aria2_builder.build()
!aria2c  --console-log-level=error -i qwen_image_aria2.txt

# Anima

In [ ]:
aria2_builder = Aria2Builder()
aria2_builder.file_path(Path("anima_aria2.txt"))

# --- ADD MODELS TO DOWNLOAD HERE ---

# Anima base model - https://huggingface.co/circlestone-labs/Anima
aria2_builder.add_url(
    "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-base-v1.0.safetensors",
    "diffusion_models",
)
aria2_builder.add_url(
    "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors",
    "text_encoders",
)
aria2_builder.add_url(
    "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors",
    "vae",
)

# Anima Turbo LoRA - https://civitai.com/models/2560840/anima-turbo-lora
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2979642?fileId=2859181",
    "loras",
)

# WAI-ANIMA alternative base model - https://civitai.red/models/2544636/wai-anima?modelVersionId=2983680
aria2_builder.add_url(
    "https://civitai.red/api/download/models/2983680?fileId=2863158",
    "diffusion_models",
)

# Velvet's Mythic Fantasy Styles - https://civitai.red/models/599757/velvets-mythic-fantasy-styles-or-flux-pony-illustrious-zit-anima?modelVersionId=2918615
aria2_builder.add_url(
    "https://civitai.red/api/download/models/2918615?fileId=2796968",
    "loras",
)
# Velvet's Mythic Fantasy Styles - https://civitai.red/models/599757/velvets-mythic-fantasy-styles-or-flux-pony-illustrious-zit-anima?modelVersionId=3016131
aria2_builder.add_url(
    "https://civitai.red/api/download/models/3016131?fileId=2895067",
    "loras",
)

# -----------------------------------

# Start download
aria2_builder.build()
!aria2c  --console-log-level=error -i anima_aria2.txt

# WAN 2.2

In [ ]:
aria2_builder = Aria2Builder()
aria2_builder.file_path(Path("wan_aria2.txt"))

# --- ADD MODELS TO DOWNLOAD HERE ---

# WAN 2.2 I2B 14B HIGH fp8_e4m3fn
aria2_builder.add_url(
    "https://huggingface.co/Kijai/WanVideo_comfy_fp8_scaled/resolve/main/I2V/Wan2_2-I2V-A14B-HIGH_fp8_e4m3fn_scaled_KJ.safetensors",
    "diffusion_models",
)
# WAN 2.2 I2B 14B LOW fp8_e4m3fn
aria2_builder.add_url(
    "https://huggingface.co/Kijai/WanVideo_comfy_fp8_scaled/resolve/main/I2V/Wan2_2-I2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors",
    "diffusion_models",
)

# WAN CLIP
aria2_builder.add_url(
    "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/umt5-xxl-enc-bf16.safetensors",
    "clip",
)
# WAN CLIP VISION
aria2_builder.add_url(
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors",
    "clip_vision",
)
# WAN NSFW CLIP VISION
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2039365?type=Model&format=SafeTensor&size=full&fp=fp16",
    "clip_vision",
)

# WAN VAE
aria2_builder.add_url(
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors",
    "vae",
)


# LoRAs
# lightx2v/Wan2.2-Lightning I2V high
aria2_builder.add_url(
    "https://huggingface.co/lightx2v/Wan2.2-Lightning/resolve/main/Wan2.2-I2V-A14B-4steps-lora-rank64-Seko-V1/high_noise_model.safetensors",
    "loras",
)
# lightx2v/Wan2.2-Lightning I2V low
aria2_builder.add_url(
    "https://huggingface.co/lightx2v/Wan2.2-Lightning/resolve/main/Wan2.2-I2V-A14B-4steps-lora-rank64-Seko-V1/low_noise_model.safetensors",
    "loras",
)

# Fingering 1.0 I2V HIGH
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2209275?type=Model&format=SafeTensor",
    "loras",
)
# Fingering 1.0 I2V LOW
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2209481?type=Model&format=SafeTensor",
    "loras",
)

# Breast play v2 I2V HIGH
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2230125?type=Model&format=SafeTensor",
    "loras",
)
# Breast play v2 I2V LOW
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2230133?type=Model&format=SafeTensor",
    "loras",
)

#M issionary I2V HIGH
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2098405?type=Model&format=SafeTensor",
    "loras",
)
# Missionary I2V LOW
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2098396?type=Model&format=SafeTensor",
    "loras",
)

# DR34MJOB Double/Single/Handy Blowjob I2V HIGH
# Keywords: bl0wj0b d0ubl3_bj
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2235299?type=Model&format=SafeTensor",
    "loras",
)
# DR34MJOB Double/Single/Handy Blowjob I2V LOW
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2235288?type=Model&format=SafeTensor",
    "loras",
)

# DR34ML4Y All-in-One NSFW I2V HIGH
# Keywords: m15510n4ry bl0wj0b d0ubl3_bj d0gg1e c0wg1rl
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2176505?type=Model&format=SafeTensor",
    "loras",
)
# DR34ML4Y All-in-One NSFW I2V LOW
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2190476?type=Model&format=SafeTensor",
    "loras",
)

# Anime Cumshot I2V HIGH
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2116008?type=Model&format=SafeTensor",
    "loras",
)
# Anime Cumshot I2V LOW
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2116027?type=Model&format=SafeTensor",
    "loras",
)

# Deepthroat v1.0 I2V HIGH
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2152516?type=Model&format=SafeTensor",
    "loras",
)
# Deepthroat v1.0 I2V LOW
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2152583?type=Model&format=SafeTensor",
    "loras",
)

# Cowgirl v1 I2V HIGH c0wg1rl + r3v3rs3_c0wg1rl
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2156392?type=Model&format=SafeTensor",
    "loras",
)
# Cowgirl v1 I2V LOW
aria2_builder.add_url(
    "https://civitai.com/api/download/models/2156435?type=Model&format=SafeTensor",
    "loras",
)









# -----------------------------------

# Start download
aria2_builder.build()
!aria2c  --console-log-level=error -i wan_aria2.txt

## MMAudio

In [ ]:
aria2_builder = Aria2Builder()
aria2_builder.file_path(Path("mmaudio_aria2.txt"))

# --- ADD MODELS TO DOWNLOAD HERE ---

aria2_builder.add_url(
    "https://huggingface.co/Kijai/MMAudio_safetensors/resolve/main/apple_DFN5B-CLIP-ViT-H-14-384_fp16.safetensors",
    "mmaudio",
)
# Base model
aria2_builder.add_url(
    "https://huggingface.co/Kijai/MMAudio_safetensors/resolve/main/mmaudio_large_44k_v2_fp16.safetensors",
    "mmaudio",
)
# NSFW model
aria2_builder.add_url(
    "https://huggingface.co/phazei/NSFW_MMaudio/resolve/main/mmaudio_large_44k_nsfw_gold_8.5k_final_fp16.safetensors",
    "mmaudio",
)


aria2_builder.add_url(
    "https://huggingface.co/Kijai/MMAudio_safetensors/resolve/main/mmaudio_synchformer_fp16.safetensors",
    "mmaudio",
)
aria2_builder.add_url(
    "https://huggingface.co/Kijai/MMAudio_safetensors/resolve/main/mmaudio_vae_44k_fp16.safetensors",
    "mmaudio",
)

# -----------------------------------

# Start download
aria2_builder.build()
!aria2c  --console-log-level=error -i mmaudio_aria2.txt